# MT Hexapod current noise at faults

This notebook will search timestamps for M2Hex faults
and analyze the current noise when it faults.\
We will only consider Faults with error codes equal to 1
For a robust analysis, this notebook anaylzes since the start of observations in April

Noise being the standard deviation of the current in the motors, given by the EfdClient


## General Data

First, lest stablish some general variables.

We'll start querying the last month, then we can change to what we see fit.\
Another important variable is the time window where the fault(s) happens.

In [ ]:
#Strut           1    2    3    4    5    6
#motorCurrent    0    1    2    3    4    5  

#I will query from around the first days in RTV
day_start = 20250630
day_end = 20250731


n_struts = 6 #Number of struts
errorCode = 1 #Will only consider errorCode 1 as Faults
salIndex = 2 #M2Hexapods (maybe make it to search for CamHex faults?)

#Upper and lower time difference from the fault
delta_start = 45
delta_end = 10

## Query the timestamps

Now we need to know when do these faults happened

In [ ]:
from astropy.time import Time, TimeDelta
from statistics import median
import pandas as pd
import matplotlib.pyplot as plt

from lsst.summit.utils.efdUtils import getDayObsEndTime, getDayObsStartTime, getEfdData
from lsst_efd_client import EfdClient

In [ ]:
#Create the client for InfluxQL
client = EfdClient('usdf_efd')

In [ ]:
start_time = getDayObsStartTime(day_start)
end_time = getDayObsEndTime(day_end)

#Query Hexapod faults
df_timestamps = getEfdData(
        client=client,
        topic='lsst.sal.MTHexapod.logevent_errorCode',
        columns=['errorCode', 'errorReport', 'salIndex',],
        begin=start_time,
        end=end_time,
    )

#Know which Timestamps caused errorCode = 1 Faults
timestamps_errorCode = df_timestamps[df_timestamps['errorCode'] == errorCode].index
#print(timestamps_errorCode) #Uncomment to see the timestamps

## Analysis of Current Noise

Now that we have the timestamps, we can query these timestamps and analyze the noise of the current in the proximity of these faults.

In [ ]:
def fcnq(motor, delta_start, delta_end, timestamp):
    #Fault Current Noise Query
    delta_start = delta_start #Non-iterable
    delta_end = delta_end #Non-iterable
    i = motor #Iterable
    fault_timestamp = timestamp #Iterable
    
    #This function should suffice for one or multiple timestamps
    #Convert index strings to datetime object and then astropytime
    t0 = Time(pd.to_datetime(fault_timestamp), scale='utc')

    #Astropy deltas
    delta1 = TimeDelta(delta_start, format='sec') #-45 seconds
    delta2 = TimeDelta(delta_end, format='sec')   #+10 seconds

    #Add the deltas, then transform to InfluxQL readable
    start_time = (t0 - delta1).to_value('isot', subfmt='date_hms') + 'Z'
    end_time = (t0 + delta2).to_value('isot', subfmt='date_hms') + 'Z'

    query = f'''SELECT stddev("motorCurrent{i}") AS "stddev_motorCurrent{i}"  
            FROM "efd"."autogen"."lsst.sal.MTHexapod.electrical" 
            WHERE time > '{start_time}' AND time < '{end_time}'
            GROUP BY time(1s) FILL(null) '''

    return(query)

In [ ]:
#Create lists to append values
median_val_0 = []
median_val_1 = []
median_val_2 = []
median_val_3 = []
median_val_4 = []
median_val_5 = []
timestamp_list = []

#Cycle through timestamps and motors, then append each to their respective list
for timestamp in timestamps_errorCode: #TimeStamps in the timespan queried
    timestamp_list.append(timestamp) #Append timestamp once per cycle
    
    for n_motor in range(n_struts): #From 0 to 5, number of struts/motors
        df_currentmotor = await client.influx_client.query(
        fcnq(n_motor, delta_start, delta_end, timestamp)
        )
        median = df_currentmotor[f'stddev_motorCurrent{n_motor}'].median()
        #print(f'The median of noise for strut id {n_motor} on timestamp {timestamp} is:{median}')
        if n_motor == 0:
            median_val_0.append(median)
        if n_motor == 1:
            median_val_1.append(median)
        if n_motor == 2:
            median_val_2.append(median)
        if n_motor == 3:
            median_val_3.append(median)
        if n_motor == 4:
            median_val_4.append(median)
        if n_motor == 5:
            median_val_5.append(median)
        

In [ ]:
#Now, we have to convert each list and timestamp list to data frames
#Converting each timestamp column into time readable by the plot

df_median_0 = pd.DataFrame({
    'Time': timestamp_list,
    'Median stddev 0': median_val_0
})
df_median_0["Time"] = pd.to_datetime(df_median_0["Time"]) 
df_median_0.set_index("Time", inplace=True)

df_median_1 = pd.DataFrame({
    'Time': timestamp_list,
    'Median stddev 1': median_val_1
})
df_median_1["Time"] = pd.to_datetime(df_median_1["Time"]) 
df_median_1.set_index("Time", inplace=True)

df_median_2 = pd.DataFrame({
    'Time': timestamp_list,
    'Median stddev 2': median_val_2
})
df_median_2["Time"] = pd.to_datetime(df_median_2["Time"]) 
df_median_2.set_index("Time", inplace=True)

df_median_3 = pd.DataFrame({
    'Time': timestamp_list,
    'Median stddev 3': median_val_3
})
df_median_3["Time"] = pd.to_datetime(df_median_3["Time"])
df_median_3.set_index("Time", inplace=True)

df_median_4 = pd.DataFrame({
    'Time': timestamp_list,
    'Median stddev 4': median_val_4
})
df_median_4["Time"] = pd.to_datetime(df_median_4["Time"])
df_median_4.set_index("Time", inplace=True)

df_median_5 = pd.DataFrame({
    'Time': timestamp_list,
    'Median stddev 5': median_val_5
})
df_median_5["Time"] = pd.to_datetime(df_median_5["Time"])
df_median_5.set_index("Time", inplace=True)


In [ ]:
#We can finally plot them
#Uncomment to save

ax = df_median_0.plot.area(figsize=(15,5), alpha = 0.5, color = 'cyan')
df_median_1.plot.area(ax = ax, alpha = 0.5, color = 'blue' )
df_median_2.plot.area(ax = ax, alpha = 0.5, color = 'violet')
df_median_3.plot.area(ax = ax, alpha = 0.5, color = 'purple')
df_median_4.plot.area(ax = ax, alpha = 0.5, color = 'red')
df_median_5.plot.area(ax = ax, alpha = 0.5, color = 'orange')

plt.ylim(0,3)
plt.grid(True)
#plt.savefig('Median_stddev_struts.png', dpi=400)
plt.show()

With this plot we can visually see that the possible main problem is the Strut 6, as the Current Noise is usually higher prior to a fault.